In [ ]:
# =============================================================================
# Antibiotic Resistance Prediction using Proximity, Frequency, and G-score
# =============================================================================
# 
# This notebook trains and evaluates Random Forest classifiers on mutation-level
# features (mutation frequency, 3D structural proximity to known resistance-conferring
# mutations, and g-scores) to predict antibiotic resistance in Mycobacterium tuberculosis.
# It includes baseline modeling,combined modeling and predictions for uncertain mutations (Category 3).

In [1]:
## load dependencies
import pandas as pd, numpy as np
from evcouplings.compare import DistanceMap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score,roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay)
from scipy import stats
import os,json,re
from pathlib import Path
from sklearn.model_selection import GridSearchCV
import seaborn as sns
import matplotlib.pyplot as plt
from utils import *

## data preprocessing

In [2]:
# ----------------------------
# STEP 1: Load and Inspect Dataset
# ----------------------------
# new_data: sanjana's data
final_df = pd.read_csv("../data/new_data/all_proteins_freq_proximity_gscore.csv")
print("Unique genes:", np.unique(final_df['gene']))
print("Data shape:", final_df.shape)
final_df['one_letter_mutation'] = final_df['one_letter_mutation'].str.replace("p.", "").str.strip()


Unique genes: ['Rv0678' 'Rv1258c' 'Rv1979c' 'Rv2983' 'ahpC' 'atpE' 'clpC1' 'ddn' 'eis'
 'embA' 'embB' 'embC' 'ethA' 'fbiA' 'fbiB' 'fbiC' 'fgd1' 'gid' 'gyrA'
 'gyrB' 'inhA' 'katG' 'mmpL5' 'mmpS5' 'mshA' 'panD' 'pepQ' 'pncA' 'rplC'
 'rpoB' 'rpsL' 'tlyA' 'ubiA' 'whiB7']
Data shape: (9365, 15)


In [3]:
# STEP 2: Basic Preprocessing
# ----------------------------
# Separate Category 3 entries (uncertain mutations)
category_3_data = final_df[final_df['confidence'] == "3) Uncertain significance"]
print(category_3_data.shape)

# Retain only labeled data (non-Category 3) for supervised learning
label_data = final_df[final_df['confidence'] != "3) Uncertain significance"]

# Target conversion to binary
label_data['binary_confidence'] = label_data['confidence'].map({
    "1) Assoc w R": 1,
    "2) Assoc w R - Interim": 1,
    "4) Not assoc w R - Interim": 0,
    "5) Not assoc w R": 0
})

# Drop missing targets
label_data = label_data[label_data['binary_confidence'].notna()]

#some debugging steps
print(label_data['confidence'].value_counts())
print(np.unique(label_data['confidence']))

# Identify rows where the 'frequency' column is NaN
nan_frequency_rows = label_data[label_data['frequency'].isna()]

diverse_sample_mutation_table = label_data.groupby('frequency').first().reset_index()
diverse_sample_mutation_table.head()

(8697, 15)
confidence
1) Assoc w R                  397
2) Assoc w R - Interim        192
4) Not assoc w R - Interim     47
5) Not assoc w R               32
Name: count, dtype: int64
['1) Assoc w R' '2) Assoc w R - Interim' '4) Not assoc w R - Interim'
 '5) Not assoc w R']


/tmp/ipykernel_855632/3485678079.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label_data['binary_confidence'] = label_data['confidence'].map({


,frequency,drug,gene,variant,confidence,one_letter_mutation,phenotype,position,Proximity_1D,Nearest_1D_Index,Proximity_to_R_Conferring,Nearest_Mutation_Index,G_score,WHO_Adjusted_Position,Proximity_to_R_Conferring_zeroed,binary_confidence
0,0.000000,Clofazimine,Rv0678,Rv0678_p.Met146Thr,2) Assoc w R - Interim,M146T,Resistant,146,4.0,142.0,2.870637,142.0,-0.455283,22.0,2.870637,1
1,0.004854,Ethambutol,embB,embB_p.Arg213Gln,5) Not assoc w R,R213Q,Susceptible,213,83.0,296.0,7.415558,497.0,1.143737,NaN,7.415558,0
2,0.007299,Ethambutol,embA,embA_p.Ala201Thr,4) Not assoc w R - Interim,A201T,Susceptible,201,407.0,608.0,64.663760,608.0,1.797703,NaN,64.663760,0
3,0.009091,Clofazimine,Rv1979c,Rv1979c_p.Asp286Gly,4) Not assoc w R - Interim,D286G,Susceptible,286,123.0,409.0,47.945507,409.0,-0.146128,NaN,47.945507,0
4,0.009569,Moxifloxacin,gyrB,gyrB_p.Pro94Leu,4) Not assoc w R - Interim,P94L,Susceptible,94,352.0,446.0,54.035292,447.0,-1.423563,NaN,54.035292,0


In [4]:

feature_sets = {
    "Proximity": ['Proximity_to_R_Conferring'],
    "Proximity_zeroed": ["Proximity_to_R_Conferring_zeroed"],
    "Proximity_1d":["Proximity_1D"],
    "G-score": ['G_score'],
    "G-score + Proximity_3D": ['G_score','Proximity_to_R_Conferring'],
    "G-score + Proximity_1D": ['G_score','Proximity_1D']

}


## helper codes

In [5]:
def normalize_variant(variant):
    """Converts rpoB_p.Ser450Leu → rpoB_S450L (1-letter style)"""
    if pd.isna(variant):
        return None
    match = re.match(r'(\w+)_p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', variant)
    if match:
        gene, wt, pos, mut = match.groups()
        wt1 = wt[0]
        mut1 = mut[0]
        return f"{gene}_{wt1}{pos}{mut1}"
    return variant  # fallback for non-matching strings

In [6]:
def merge_cols(df_2021, df_2023):
    merge_cols = ["gene", "variant", "one_letter_mutation"]

    merged_df = pd.merge(
        df_2021,
        df_2023,
        on=merge_cols,
        suffixes=('_2021', '_2023')
    )

    # Clean confidence fields
    merged_df['confidence_2021'] = merged_df['confidence_2021'].str.strip()
    merged_df['confidence_2023'] = merged_df['confidence_2023'].str.strip()

    # Filter unchanged Category 3
    unchanged_category_3 = merged_df[
        (merged_df['confidence_2021'] == "3) Uncertain significance") &
        (merged_df['confidence_2023'] == "3) Uncertain significance")
    ]

    print(f"Still Category 3 in both datasets: {unchanged_category_3.shape[0]}")
    display(unchanged_category_3[["gene", "variant", "confidence_2021", "confidence_2023"]].head())
    return merged_df


In [7]:
def reclassified_variants(df_2021, df_2023):
    # Normalize 2023
    merged_df=merge_cols(df_2021, df_2023)
    # Step 1: Filter Updated Variants
    # Remove unchanged "3) Uncertain significance" cases
    filtered_df = merged_df[
        ~((merged_df['confidence_2021'] == "3) Uncertain significance") &
        (merged_df['confidence_2023'] == "3) Uncertain significance"))
    ]
    # now we are left with residues that have changed confidence levels
    # Find entries where confidence values differ
    updated_confidence = filtered_df[
        filtered_df['confidence_2021'] != filtered_df['confidence_2023']
    ].drop_duplicates()

    # Print basic stats
    print("Changes in confidence labels between two datasets:")
    print(filtered_df['confidence_2021'].value_counts())
    print(filtered_df['confidence_2023'].value_counts())
    print('there are more resistant mutations in 2023')


    # Step 2: Get mutations reclassified from Category 3 → confident
    # Must have been 3 in 2021, but something else in 2023
    confidence_3_changed = updated_confidence[
        (updated_confidence['confidence_2021'] == "3) Uncertain significance") &
        (updated_confidence['confidence_2023'] != "3) Uncertain significance")
    ].copy()


    print("New 2023 confidence label breakdown:")
    confidence_3_dedup = confidence_3_changed.drop_duplicates(subset=['gene', 'one_letter_mutation'])
    print(confidence_3_dedup['confidence_2023'].value_counts())
    print(f"\nVariants reclassified from uncertain to confident from 2021 to 2023: {confidence_3_dedup.shape[0]}")

    # Drop duplicates from both before merge
    label_data_dedup = label_data.drop_duplicates(subset=['gene', 'one_letter_mutation'])
    confidence_3_dedup = confidence_3_changed.drop_duplicates(subset=['gene', 'one_letter_mutation'])

    # Merge again
    reclassified_eval = pd.merge(
        label_data_dedup,
        confidence_3_dedup[['gene', 'one_letter_mutation', 'confidence_2023']],
        on=['gene', 'one_letter_mutation'],
        how='inner'
    )
    print(f"\nReclassified evaluation dataset shape: {reclassified_eval.shape}")

    # number of reclassified mutations
    gene_counts = confidence_3_dedup['gene'].value_counts().reset_index()
    gene_counts.columns = ['gene', 'reclassified_count']
    print(gene_counts)

    gene_level_distribution = (confidence_3_dedup
        .groupby(['gene','confidence_2023'])
        .size()
        .reset_index(name='count')
        .sort_values(by= ['gene','count'], ascending=[True, False]))


    print(gene_level_distribution)
    return reclassified_eval

In [8]:
def plot_feature_distribution(reclassified_eval, model_name, label):
    import os
    reclassified_eval['correct_str'] = reclassified_eval['correct'].map({True: "Correct", False: "Incorrect"})

    features = [
        ("G_score", "G_score Distribution"),
        ("Proximity_to_R_Conferring", "Proximity_to_R_Conferring"),
        ("Proximity_1D", "Proximity_1D")
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

    for ax, (feature, title) in zip(axes, features):
        sns.boxplot(data=reclassified_eval, x='correct_str', y=feature, ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Prediction Correctness")
        ax.set_ylabel(feature)

    fig.suptitle(f"Feature Distribution by Prediction Correctness\nModel: {model_name} | Feature Set: {label}", fontsize=16, y=1.05)
    plt.tight_layout()

    # Save figure
    filename = f"../data/results/feature_dist/{label.replace(' ', '_')}_{model_name.replace(' ', '_')}_feature_distribution.png"
    fig.savefig(filename, bbox_inches="tight")
    print(f"Saved boxplots to: {filename}")
    plt.close(fig)

def plot_confusion_matrix(cm, label, model_name):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Sus", "Res"], yticklabels=["Sus", "Res"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix ({label} - {model_name})")
    plt.tight_layout()

    filename = f"../data/results/confusion_matrix/{label.replace(' ', '_')}_{model_name.replace(' ', '_')}_confusion_matrix.png"
    fig.savefig(filename)
    print(f"Saved confusion matrix to: {filename}")
    plt.close(fig)


def plot_false_positives_by_gene(df, model_name, label):
    fp = df[(df['true_label'] == 0) & (df['model_prediction'] == 1)]
    print(f"Total false positives (affecting specificity): {len(fp)}")
    gene_counts = fp['gene'].value_counts()

    if not gene_counts.empty:
        fig, ax = plt.subplots(figsize=(10, 6))

        sns.barplot(
            y=gene_counts.index,
            x=gene_counts.values,
            hue=gene_counts.index,         # Explicitly set hue
            palette='Blues_d',
            legend=False,                  # Avoid redundant legend
            ax=ax
        )

        ax.set_title("False Positives by Gene (Specificity Errors)")
        ax.set_xlabel("Number of False Positives")
        ax.set_ylabel("Gene")
        plt.tight_layout()

        out_dir = "../data/results/false_positive_by_gene"
        os.makedirs(out_dir, exist_ok=True)
        filename = f"{out_dir}/{label.replace(' ', '_')}_{model_name.replace(' ', '_')}_false_positive_genes.png"
        fig.savefig(filename)
        print(f"Saved false positive gene barplot to: {filename}")
        plt.close(fig)



In [15]:
def evaluate_model(model, model_name, label, numeric_columns, reclassified_eval):
    # ─── Prepare Features ─────────────────────────────────────────────
    reclassified_eval[numeric_columns] = reclassified_eval[numeric_columns].fillna(0)
    reclassified_eval['true_label'] = reclassified_eval['confidence_2023'].map(label_map)
    X = reclassified_eval[numeric_columns]
    y_true = reclassified_eval['true_label'].values

    # ─── Prediction ───────────────────────────────────────────────────
    if model_name == "Logistic Regression":
        y_probs = model.predict_proba(X)[:, 1]
        threshold = 0.6
        y_pred = (y_probs >= threshold).astype(int)
    else:
        y_pred = model.predict(X)

    # Add predictions to df
    reclassified_eval['model_prediction'] = y_pred
    reclassified_eval['correct'] = y_pred == y_true

    # ─── Metrics ──────────────────────────────────────────────────────
    accuracy = reclassified_eval['correct'].mean() * 100

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    try:
        y_probs = model.predict_proba(X)[:, 1]
        roc_auc = roc_auc_score(y_true, y_probs)
    except AttributeError:
        roc_auc = float('nan')

    report_dict = classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=['Susceptible', 'Resistant'],
        output_dict=True,
        zero_division=0
    )

    # ─── Summary Table ────────────────────────────────────────────────
    results_summary.append({
        'Feature Set': label,
        'Model': model_name,
        'Accuracy (%)': accuracy,
        'ROC AUC': 100 * roc_auc,
        'Sensitivity (Res)': 100 * sensitivity,
        'Specificity (Sus)': 100 * specificity,
        'Precision (Res)': 100 * report_dict['Resistant']['precision'],
        'Recall (Res)': 100 * report_dict['Resistant']['recall'],
        'F1 (Res)': 100 * report_dict['Resistant']['f1-score'],
        'Precision (Sus)': 100 * report_dict['Susceptible']['precision'],
        'Recall (Sus)': 100 * report_dict['Susceptible']['recall'],
        'F1 (Sus)': 100 * report_dict['Susceptible']['f1-score'],
        'Support (Res)': int(report_dict['Resistant']['support']),
        'Support (Sus)': int(report_dict['Susceptible']['support']),
        'Incorrect Predictions': (~reclassified_eval['correct']).sum()
    })

    # ─── Visuals ──────────────────────────────────────────────────────
    plot_confusion_matrix(cm, label, model_name)
    plot_feature_distribution(reclassified_eval, model_name, label)
    plot_false_positives_by_gene(reclassified_eval, model_name, label)

    # # ─── Optional: View Incorrect Predictions ─────────────────────────
    # incorrect_preds = reclassified_eval[~reclassified_eval['correct']]
    # display(incorrect_preds[['gene', 'one_letter_mutation', 'confidence_2023', 'true_label', 'model_prediction']].sort_values(by='gene'))


In [10]:
def save_incorrect_preds(model_name, df, label):
    incorrect = df[~df['correct']].copy()
    filename = f"../data/results/false_positive_csv/{label.replace(' ', '_')}_{model_name.replace(' ', '_')}_false_positives.csv"
    incorrect.to_csv(filename, index=False)
    print(f"Saved {model_name} incorrect predictions to: {filename}")


In [11]:
# def find_invalid_keys(obj, path="root"):
#     if isinstance(obj, dict):
#         for k, v in obj.items():
#             if not isinstance(k, (str, int, float, bool, type(None))):
#                 print(f"[Invalid Key] Path: {path} → Key: {k} (type: {type(k).__name__})")
#             find_invalid_keys(v, path=f"{path}['{k}']")
#     elif isinstance(obj, list):
#         for i, item in enumerate(obj):
#             find_invalid_keys(item, path=f"{path}[{i}]")


# # Check before saving
# find_invalid_keys(results)

# def convert_keys_to_str(obj):
#     if isinstance(obj, dict):
#         return {str(k): convert_keys_to_str(v) for k, v in obj.items()}
#     elif isinstance(obj, list):
#         return [convert_keys_to_str(i) for i in obj]
#     else:
#         return obj
# cleaned_results = convert_keys_to_str(results)

# with open("../data/results/sanjana_logreg_combined_results.json", "w") as f:
#     for label in cleaned_results:
#         for key in cleaned_results[label]:
#             if not isinstance(key, (str, int, float, bool, type(None))):
#                 print("Invalid key type:", key, type(key))
#     json.dump(cleaned_results, f, indent=4, default=str)

# print("\nAll model combinations evaluated and saved.")

# def generate_markdown_table(results_dict):
#     table_md = "| Model | AUC (avg) | Precision | Sensitivity | Specificity\n"
#     table_md += "|--------|------------|-----------|-----------|-----------|\n"

#     for model_name, metrics in results_dict.items():

#         prec = f"{metrics.get('precision', 0) * 100:.1f}%"

#         roc_auc = metrics.get('roc_auc', None)
#         sensitivity = f"{metrics.get('sensitivity', 0) * 100:.1f}%"
#         specificity = f"{metrics.get('specificity', 0) * 100:.1f}%"
#         if isinstance(roc_auc, dict):
#             # Average over classes
#             auc_val = np.mean(list(roc_auc.values()))
#         else:
#             auc_val = roc_auc

#         auc_str = f"{auc_val * 100:.1f}%" if auc_val is not None else "N/A"

#         table_md += f"| {model_name} | {auc_str} | {prec} | {sensitivity} | {specificity} |\n"

#     return table_md

# markdown_output = generate_markdown_table(cleaned_results)
# print(markdown_output)
# ## this is the result prior to hyperparam tuning

## run hyperparam tuning for random forest and logreg

In [15]:


def hyperparameter_tuning(X_train, y_train, X_test, model_type='random_forest'):
    if model_type == 'random_forest':
        model = RandomForestClassifier(random_state=42)
        param_grid = {
            'n_estimators': [100, 200, 500],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2'],
            'class_weight': ['balanced', 'balanced_subsample']
        }

    elif model_type == 'logistic_regression':
        model = LogisticRegression(max_iter=1000, solver='liblinear') 
        param_grid = {
            'penalty': ['l1', 'l2'],
            'C': [0.01, 0.1, 1, 10, 100],
            'class_weight': [None, 'balanced']
        }

    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    # Grid Search
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        scoring='roc_auc',  
        verbose=1
    )

    grid_search.fit(X_train, y_train)
    print(f"\nBest parameters for {model_type}:")
    print(grid_search.best_params_)

    # Best model
    best_model = grid_search.best_estimator_

    # Predict
    y_pred_test = best_model.predict(X_test)
    y_prob_test = best_model.predict_proba(X_test)

    return best_model, y_pred_test, y_prob_test


In [16]:


# Define save directory
param_dir = Path("../models")
param_dir.mkdir(parents=True, exist_ok=True)

# Initialize results list
results_summary = []

# Define which models to run
model_types = ['random_forest', 'logistic_regression']

for label, numeric_columns in feature_sets.items():
    print(f"\n=== Feature Set: {label} ===")
    
    # Step 1: Feature selection
    label_data[numeric_columns] = label_data[numeric_columns].fillna(0)

    # Step 2: Split
    X_train, X_test, y_train, y_test, train_indices, test_indices = data_split(label_data, numeric_columns)

    for model_type in model_types:
        print(f"--- Training {model_type.replace('_', ' ').title()} ---")

        # Step 3: Tune & Train
        trained_model, y_pred_test, y_prob_test = hyperparameter_tuning(
            X_train, y_train, X_test, model_type=model_type
        )

        # Save best parameters
        param_path = param_dir / f"{model_type}_best_params_{label}.json"
        with open(param_path, "w") as f:
            json.dump(trained_model.get_params(), f, indent=4)



=== Feature Set: Proximity ===
--- Training Random Forest ---
Fitting 5 folds for each of 432 candidates, totalling 2160 fits


/work/pi_annagreen_umass_edu/mahbuba/protein/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



Best parameters for random_forest:
{'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
--- Training Logistic Regression ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters for logistic_regression:
{'C': 0.01, 'class_weight': None, 'penalty': 'l2'}

=== Feature Set: Proximity_zeroed ===
--- Training Random Forest ---
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

Best parameters for random_forest:
{'class_weight': 'balanced_subsample', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 500}
--- Training Logistic Regression ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters for logistic_regression:
{'C': 0.01, 'class_weight': None, 'penalty': 'l2'}

=== Feature Set: Proximity_1d ===
--- Training Random Forest ---
Fitting 5 folds for each of 432 candidates, totalling 

/work/pi_annagreen_umass_edu/mahbuba/protein/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



Best parameters for random_forest:
{'class_weight': 'balanced_subsample', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 500}
--- Training Logistic Regression ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters for logistic_regression:
{'C': 1, 'class_weight': None, 'penalty': 'l2'}


## run on logreg and rf best params

In [12]:
# Load the 2021 and 2023 datasets

file_2021 = "../data/derived_features/all_proteins_freq_proximity_gscore.csv"
df_2021 = pd.read_csv(file_2021)
df_2023=final_df
df_2023["variant"] = df_2023["variant"].apply(normalize_variant)

# Map binary labels as used in training
label_map = {
    "1) Assoc w R": 1,
    "2) Assoc w R - Interim": 1,
    "4) Not assoc w R - Interim": 0,
    "5) Not assoc w R": 0
}
reclassified_eval = reclassified_variants(df_2021, df_2023)

Still Category 3 in both datasets: 2425


,gene,variant,confidence_2021,confidence_2023
40,rpoB,rpoB_L731P,3) Uncertain significance,3) Uncertain significance
41,rpoB,rpoB_A692T,3) Uncertain significance,3) Uncertain significance
42,rpoB,rpoB_I488V,3) Uncertain significance,3) Uncertain significance
43,rpoB,rpoB_I480V,3) Uncertain significance,3) Uncertain significance
44,rpoB,rpoB_A286V,3) Uncertain significance,3) Uncertain significance


Changes in confidence labels between two datasets:
confidence_2021
3) Uncertain significance     131
2) Assoc w R - Interim         73
1) Assoc w R                   72
5) Not assoc w R               35
4) Not assoc w R - Interim     22
Name: count, dtype: int64
confidence_2023
1) Assoc w R                  173
2) Assoc w R - Interim         81
3) Uncertain significance      39
4) Not assoc w R - Interim     20
5) Not assoc w R               20
Name: count, dtype: int64
there are more resistant mutations in 2023
New 2023 confidence label breakdown:
confidence_2023
1) Assoc w R                  59
2) Assoc w R - Interim        27
4) Not assoc w R - Interim    12
5) Not assoc w R               6
Name: count, dtype: int64

Variants reclassified from uncertain to confident from 2021 to 2023: 104

Reclassified evaluation dataset shape: (104, 17)
      gene  reclassified_count
0      gid                  22
1     pncA                  21
2     katG                  14
3     ethA             

In [13]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [16]:
# Save/load directory for best params
param_dir = Path("../models")
param_dir.mkdir(parents=True, exist_ok=True)

# Initialize results list
results_summary = []
category_3_results = {}
for label, numeric_columns in feature_sets.items():
    print(f"\n=== Running models for feature set: {label} ===")

    # ─── Step 1: Feature selection ───────────────────────────────
    label_data[numeric_columns] = label_data[numeric_columns].fillna(0)

    # ─── Step 2: Train-test split ────────────────────────────────
    X_train, X_test, y_train, y_test, train_indices, test_indices = data_split(label_data, numeric_columns)

    # ─── Step 3: Load RF best params ─────────────────────────────
    rf_param_path = param_dir / f"random_forest_best_params_{label}.json"
    with open(rf_param_path) as f:
        rf_best_params = json.load(f)
    rf_best_params["random_state"] = 42

    # ─── Step 4: Train + Evaluate RF ─────────────────────────────
    rf_model = RandomForestClassifier(**rf_best_params)
    rf_model.fit(X_train, y_train)
    evaluate_model(rf_model, "Random Forest", label, numeric_columns, reclassified_eval)
    save_incorrect_preds("Random Forest", reclassified_eval.copy(), label)
    # Predict Category 3 with RF
    (misclassified_original_indices_rf,misclassified_samples_rf,conf0_count_rf,conf0_pct_rf,conf1_count_rf,conf1_pct_rf) = predict_category_3(category_3_data,rf_model,final_df,numeric_columns,y_test,rf_model.predict(X_test),test_indices)
    category_3_results[f"{label}_RF"] = {
        "misclassified_indices": misclassified_original_indices_rf,
        "category_3_predictions": {
            "resistant_count": conf0_count_rf,
            "susceptible_count": conf1_count_rf,
            "resistant_percentage": conf0_pct_rf,
            "susceptible_percentage": conf1_pct_rf
        }
    }
    # ─── Step 5: Load or define LogReg best params ─────────────── hyperparam tuned log regression performs worse
    # logreg_param_path = param_dir / f"logistic_regression_best_params_{label}.json"
    # if logreg_param_path.exists():
    #     with open(logreg_param_path) as f:
    #         logreg_best_params = json.load(f)
    #     logreg_best_params["random_state"] = 42
    #     logreg_model = LogisticRegression(**logreg_best_params)
    # else:
    #     logreg_model = LogisticRegression(max_iter=1000, random_state=42)
    logreg_model = LogisticRegression(max_iter=1000, random_state=42)
    # ─── Step 6: Train + Evaluate LogReg ─────────────────────────
    logreg_model.fit(X_train, y_train)
    evaluate_model(logreg_model, "Logistic Regression", label, numeric_columns, reclassified_eval)
    save_incorrect_preds("Logistic Regression", reclassified_eval.copy(), label)
    # Predict Category 3 with LogReg
    (misclassified_original_indices_lr,misclassified_samples_lr,conf0_count_lr,conf0_pct_lr,conf1_count_lr,conf1_pct_lr) = predict_category_3(category_3_data,logreg_model,final_df,numeric_columns,y_test,logreg_model.predict(X_test),test_indices)

    category_3_results[f"{label}_LogReg"] = {
        "misclassified_indices": misclassified_original_indices_lr,
        "category_3_predictions": {
            "resistant_count": conf0_count_lr,
            "susceptible_count": conf1_count_lr,
            "resistant_percentage": conf0_pct_lr,
            "susceptible_percentage": conf1_pct_lr
        }
    }
# ─── Final: Save results ────────────────────────────────────────
results_df = pd.DataFrame(results_summary)

# Reorder for clarity
results_df = results_df[
    ['Feature Set', 'Model', 'Accuracy (%)','ROC AUC', 'Sensitivity (Res)', 'Specificity (Sus)',
     'Precision (Res)', 'Recall (Res)', 'F1 (Res)',
     'Precision (Sus)', 'Recall (Sus)', 'F1 (Sus)',
     'Support (Res)', 'Support (Sus)', 'Incorrect Predictions']
]

print("\nSummary of Model Performance Across Feature Sets:")
print(results_df.round(2))

results_df.to_csv("../data/results/reclassified_feature_set_summary.csv", index=False)
# Convert all indices to list for JSON serialization
for key in category_3_results:
    # Convert misclassified_indices to list if needed
    if isinstance(category_3_results[key].get("misclassified_indices"), (pd.Series, pd.Index)):
        category_3_results[key]["misclassified_indices"] = category_3_results[key]["misclassified_indices"].tolist()

    # Ensure nested values in category_3_predictions are basic Python types
    predictions = category_3_results[key].get("category_3_predictions", {})
    for pred_key, pred_value in predictions.items():
        if isinstance(pred_value, (np.integer, np.floating, np.bool_)):
            category_3_results[key]["category_3_predictions"][pred_key] = pred_value.item()

# Save category 3 predictions summary
with open("../data/results/category_3_summary.json", "w") as f:
    json.dump(category_3_results, f, indent=4)



=== Running models for feature set: Proximity ===
Saved confusion matrix to: ../data/results/confusion_matrix/Proximity_Random_Forest_confusion_matrix.png
Saved boxplots to: ../data/results/feature_dist/Proximity_Random_Forest_feature_distribution.png
Total false positives (affecting specificity): 3
Saved false positive gene barplot to: ../data/results/false_positive_by_gene/Proximity_Random_Forest_false_positive_genes.png
Saved Random Forest incorrect predictions to: ../data/results/false_positive_csv/Proximity_Random_Forest_false_positives.csv
Saved confusion matrix to: ../data/results/confusion_matrix/Proximity_Logistic_Regression_confusion_matrix.png
Saved boxplots to: ../data/results/feature_dist/Proximity_Logistic_Regression_feature_distribution.png
Total false positives (affecting specificity): 4
Saved false positive gene barplot to: ../data/results/false_positive_by_gene/Proximity_Logistic_Regression_false_positive_genes.png
Saved Logistic Regression incorrect predictions to: 

In [17]:
results_df

,Feature Set,Model,Accuracy (%),ROC AUC,Sensitivity (Res),Specificity (Sus),Precision (Res),Recall (Res),F1 (Res),Precision (Sus),Recall (Sus),F1 (Sus),Support (Res),Support (Sus),Incorrect Predictions
0,Proximity,Random Forest,18.269231,64.341085,4.651163,83.333333,57.142857,4.651163,8.602151,15.463918,83.333333,26.086957,86,18,85
1,Proximity,Logistic Regression,50.000000,74.773902,44.186047,77.777778,90.476190,44.186047,59.375000,22.580645,77.777778,35.000000,86,18,52
2,Proximity_zeroed,Random Forest,50.961538,70.122739,45.348837,77.777778,90.697674,45.348837,60.465116,22.950820,77.777778,35.443038,86,18,51
3,Proximity_zeroed,Logistic Regression,70.192308,80.038760,68.604651,77.777778,93.650794,68.604651,79.194631,34.146341,77.777778,47.457627,86,18,31
4,Proximity_1d,Random Forest,17.307692,60.335917,0.000000,100.000000,0.000000,0.000000,0.000000,17.307692,100.000000,29.508197,86,18,86
5,Proximity_1d,Logistic Regression,52.884615,88.404393,44.186047,94.444444,97.435897,44.186047,60.800000,26.153846,94.444444,40.963855,86,18,49
6,G-score,Random Forest,74.038462,54.392765,86.046512,16.666667,83.146067,86.046512,84.571429,20.000000,16.666667,18.181818,86,18,27
7,G-score,Logistic Regression,82.692308,82.202842,98.837209,5.555556,83.333333,98.837209,90.425532,50.000000,5.555556,10.000000,86,18,18
8,G-score + Proximity_3D,Random Forest,31.730769,63.242894,22.093023,77.777778,82.608696,22.093023,34.862385,17.283951,77.777778,28.282828,86,18,71
9,G-score + Proximity_3D,Logistic Regression,75.961538,85.432817,75.581395,77.777778,94.202899,75.581395,83.870968,40.000000,77.777778,52.830189,86,18,25


In [ ]:
## hyperparam tuned logistic regression produces degenerated results- specially for g-score (100% sensitivity) 